In [ ]:
%pip install pennylane pennylane-qiskit qiskit qiskit-ibm-runtime

In [ ]:
import torch
import torch.nn as nn
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from kaggle_secrets import UserSecretsClient # type: ignore
IBM_RUNTIME_API_KEY = UserSecretsClient().get_secret("IBM_RUNTIME_API_KEY")

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token=IBM_RUNTIME_API_KEY,
    overwrite=True,
    set_as_default=True,
)

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_cloud")

backend = service.least_busy(
    simulator=False,
    operational=True
)

In [ ]:

Re = 100

n_qubits = 4
initial_layers = 1
max_layers = 4

q_device = qml.device("default.qubit", wires=n_qubits)

In [ ]:
import time
from pennylane.qnn import TorchLayer

class QuantumLayer(nn.Module):
    _call_count = 0
    _total_time = 0.0

    def __init__(self, n_layers):
        super().__init__()
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.layer = TorchLayer(self.make_circuit(n_layers), weight_shapes) # type: ignore

    def forward(self, x):
        t0 = time.perf_counter()
        output = self.layer(x)   # (batch, n_qubits) -> (batch, n_qubits)
        t1 = time.perf_counter()

        elapsed = t1 - t0
        QuantumLayer._call_count += 1
        QuantumLayer._total_time += elapsed
        if QuantumLayer._call_count % 50 == 1:
            print(
                f"  [QuantumLayer] call #{QuantumLayer._call_count}: "
                f"{elapsed * 1000:.1f}ms total for {x.shape[0]} samples "
                f"({elapsed / x.shape[0] * 1000:.3f}ms/sample)"
            )

        return output

    @staticmethod
    def make_circuit(layers: int):
        @qml.qnode(q_device, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')

            for l in range(layers):
                for q in range(n_qubits):
                    qml.RX(weights[l, q, 0], wires=q)
                    qml.RY(weights[l, q, 1], wires=q)
                    qml.RZ(weights[l, q, 2], wires=q)

                for q in range(n_qubits - 1):
                    qml.CNOT(wires=[q, q + 1])

            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        return circuit


In [1]:
qml.drawer.use_style("black_white") # type: ignore
dummy_inputs = torch.zeros(n_qubits)
dummy_weights = torch.zeros(initial_layers, n_qubits, 3)

fig, ax = qml.draw_mpl(QuantumLayer.make_circuit(initial_layers))(dummy_inputs, dummy_weights)
plt.show()


NameError: name 'qml' is not defined

In [28]:
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device_torch)

Device: cpu


In [ ]:
class HybridQPINN(nn.Module):
    _call_count = 0

    def __init__(self, n_layers):
        super().__init__()

        self.embedding = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits)
        )

        self.quantum = QuantumLayer(n_layers)

        self.decoder = nn.Sequential(
            nn.Linear(n_qubits, 32),
            nn.Tanh(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        t0 = time.perf_counter()
        q_in = self.embedding(x)
        t1 = time.perf_counter()
        q_out = self.quantum(q_in)
        t2 = time.perf_counter()
        out = self.decoder(q_out)
        t3 = time.perf_counter()

        HybridQPINN._call_count += 1
        if HybridQPINN._call_count % 50 == 1:
            print(
                f"  [HybridQPINN] call #{HybridQPINN._call_count}: "
                f"embed={( t1 - t0) * 1000:.2f}ms  "
                f"quantum={(t2 - t1) * 1000:.1f}ms  "
                f"decode={(t3 - t2) * 1000:.2f}ms  "
                f"total={(t3 - t0) * 1000:.1f}ms"
            )

        psi = out[:,0:1]
        p = out[:,1:2]

        return psi, p


In [17]:
def velocity(model, coords):
    coords.requires_grad_(True)
    psi,_ = model(coords)

    grad = torch.autograd.grad(
        psi,
        coords,
        grad_outputs=torch.ones_like(psi),
        create_graph=True
    )[0]

    u = grad[:,1:2]
    v = -grad[:,0:1]

    return u,v

In [ ]:
def residual(model, coords):
    coords.requires_grad_(True)

    _psi, p = model(coords)
    u, v = velocity(model, coords)

    grads_u = torch.autograd.grad(u, coords, torch.ones_like(u), create_graph=True)[0]
    grads_v = torch.autograd.grad(v, coords, torch.ones_like(v), create_graph=True)[0]

    u_x = grads_u[:, 0:1]
    u_y = grads_u[:, 1:2]

    v_x = grads_v[:, 0:1]
    v_y = grads_v[:, 1:2]

    u_xx = torch.autograd.grad(u_x, coords, torch.ones_like(u_x), create_graph=True)[0][
        :, 0:1
    ]
    u_yy = torch.autograd.grad(u_y, coords, torch.ones_like(u_y), create_graph=True)[0][
        :, 1:2
    ]

    v_xx = torch.autograd.grad(v_x, coords, torch.ones_like(v_x), create_graph=True)[0][
        :, 0:1
    ]
    v_yy = torch.autograd.grad(v_y, coords, torch.ones_like(v_y), create_graph=True)[0][
        :, 1:2
    ]

    grad_p = torch.autograd.grad(p, coords, torch.ones_like(p), create_graph=True)[0]

    p_x = grad_p[:, 0:1]
    p_y = grad_p[:, 1:2]

    mx = u * u_x + v * u_y + p_x - (1 / Re) * (u_xx + u_yy)
    my = u * v_x + v * v_y + p_y - (1 / Re) * (v_xx + v_yy)

    return mx, my


def boundary_loss(model):
    pts = torch.rand(2000, 2, device=device_torch)
    u, v = velocity(model, pts)

    x = pts[:, 0:1]
    y = pts[:, 1:2]

    lid = y > 0.99
    lid_loss = torch.mean((u[lid] - 1) ** 2) + torch.mean(v[lid] ** 2)
    wall = (y < 0.01) | (x < 0.01) | (x > 0.99)
    wall_loss = torch.mean(u[wall] ** 2) + torch.mean(v[wall] ** 2)

    return lid_loss + wall_loss


In [ ]:
def adaptive_sampling(model, n_points=2000):
    # Reduced candidates: 10 000 → 2 000 to avoid running the quantum circuit
    # on a massive batch just for importance-sampling weights.
    candidate = torch.rand(2000, 2, device=device_torch)
    mx, my = residual(model, candidate)
    res = (mx**2 + my**2).detach().flatten()
    prob = res / res.sum()
    idx = torch.multinomial(prob, n_points, replacement=True)

    return candidate[idx]


In [ ]:
model = HybridQPINN(initial_layers).to(device_torch)

def count_parameters(model):

    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)

    return total_params, trainable_params

count_parameters(model)


Total parameters: 466
Trainable parameters: 466


(466, 466)

In [ ]:
import time

LOG_INTERVAL = 10   # print step breakdown every N epochs

layers = initial_layers
epoch_times = []

# reset layer-level counters whenever the training loop is (re)started
QuantumLayer._call_count = 0
QuantumLayer._total_time = 0.0
HybridQPINN._call_count = 0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    ep_start = time.perf_counter()

    t0 = time.perf_counter()
    pts = adaptive_sampling(model, 2000)
    t1 = time.perf_counter()

    mx, my = residual(model, pts)
    phys_loss = torch.mean(mx**2) + torch.mean(my**2)
    t2 = time.perf_counter()

    bc = boundary_loss(model)
    loss = phys_loss + bc
    t3 = time.perf_counter()

    optimizer.zero_grad()
    loss.backward()
    t4 = time.perf_counter()

    optimizer.step()
    t5 = time.perf_counter()

    epoch_time = t5 - ep_start
    epoch_times.append(epoch_time)

    if epoch % LOG_INTERVAL == 0:
        print(
            f"\nEpoch {epoch} | Loss {loss:.6f} | Total {epoch_time:.3f}s\n"
            f"  adaptive_sampling : {(t1 - t0):.3f}s\n"
            f"  residual          : {(t2 - t1):.3f}s\n"
            f"  boundary_loss     : {(t3 - t2):.3f}s\n"
            f"  backward          : {(t4 - t3):.3f}s\n"
            f"  optimizer.step    : {(t5 - t4):.3f}s"
        )

    # depth scaling
    if epoch > 0 and epoch % 2000 == 0 and initial_layers < max_layers:
        layers += 1
        print("Increasing quantum depth to", layers)

        model = HybridQPINN(layers).to(device_torch)
        QuantumLayer._call_count = 0
        QuantumLayer._total_time = 0.0
        HybridQPINN._call_count = 0
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"\nQuantumLayer total wall time across all calls: {QuantumLayer._total_time:.3f}s over {QuantumLayer._call_count} calls")


In [ ]:
grid = 60

x = np.linspace(0, 1, grid)
y = np.linspace(0, 1, grid)

X, Y = np.meshgrid(x, y)

coords = torch.tensor(np.vstack([X.flatten(), Y.flatten()]).T, dtype=torch.float32).to(device_torch)

u, v = velocity(model, coords)

U = u.detach().cpu().numpy().reshape(grid, grid)
V = v.detach().cpu().numpy().reshape(grid, grid)

plt.figure(figsize=(6, 6))

plt.streamplot(X, Y, U, V, density=2)

plt.title("Lid Driven Cavity Streamlines")

plt.show()
